# Feature Matching Benchmark (SIFT vs ORB)

SIFT와 ORB 특징점 검출기를 BF/FLANN 매처와 조합하여 성능을 비교하는 실습

In [ ]:
import cv2
import time
import numpy as np
import urllib.request
from google.colab.patches import cv2_imshow  # 코랩 전용 출력

## 이미지 로드 및 흑백 변환

In [ ]:
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
urllib.request.urlretrieve(url, 'aloeL.jpg')
img1 = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)

url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url, 'aloeR.jpg')
img2 = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

## 벤치마크 함수 정의

In [ ]:
def benchmark_full(name, detector, matcher_type='BF', ratio_threshold=0.7):
    start_time = time.time()
    kp1, des1 = detector.detectAndCompute(img1, None)
    kp2, des2 = detector.detectAndCompute(img2, None)
    detect_time = (time.time() - start_time) * 1000

    # 매처(Matcher) 설정
    if matcher_type == 'BF':
        norm = cv2.NORM_L2 if name == "SIFT" else cv2.NORM_HAMMING
        matcher = cv2.BFMatcher(norm, crossCheck=False)
    else:
        if name == "SIFT":
            index_params = dict(algorithm=1, trees=5)  # FLANN_INDEX_KDTREE
        else:
            index_params = dict(algorithm=6, table_number=6,
                                key_size=12, multi_probe_level=1)
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)

    # k-NN 매칭 및 Ratio Test
    start_time = time.time()
    matches = matcher.knnMatch(des1, des2, k=2)
    good_matches = []
    for m_n in matches:
        if len(m_n) == 2:
            m, n = m_n
            if m.distance < ratio_threshold * n.distance:
                good_matches.append(m)
    match_time = (time.time() - start_time) * 1000

    print(f"\n--- [{name} + {matcher_type}] ---")
    print(f"Detect: {detect_time:.1f}ms | Match: {match_time:.1f}ms | Goods: {len(good_matches)}")

    # 매칭 결과 출력 (상위 50개)
    res_img = cv2.drawMatches(img1, kp1, img2, kp2,
                              good_matches[:50], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    # 결과 이미지 위에 텍스트 정보 표시
    info_text = f"{name}+{matcher_type}: {len(good_matches)} matches"
    cv2.putText(res_img, info_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2_imshow(res_img)

    return kp1, kp2, good_matches

## SIFT 벤치마크 (BF + FLANN)

In [ ]:
sift = cv2.SIFT_create()

print("SIFT 벤치마크 시작...")
benchmark_full("SIFT", sift, "BF")
benchmark_full("SIFT", sift, "FLANN")

## ORB 벤치마크 (BF + FLANN)

In [ ]:
orb = cv2.ORB_create(nfeatures=2000)

print("ORB 벤치마크 시작...")
benchmark_full("ORB", orb, "BF", ratio_threshold=0.85)
benchmark_full("ORB", orb, "FLANN", ratio_threshold=0.85)